In [116]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import string

from urllib.parse import urlparse,parse_qs

from plotly import graph_objects as go
from plotly.subplots import make_subplots

import tldextract
import pycountry	



## 读取数据

In [4]:
import os
os.getcwd()

os.chdir("malicious-url-detect-project")

In [5]:
data = pd.read_csv('balanced_urls.csv')

In [6]:
data.head()

,url,label,result
0,https://www.google.com,benign,0
1,https://www.youtube.com,benign,0
2,https://www.facebook.com,benign,0
3,https://www.baidu.com,benign,0
4,https://www.wikipedia.org,benign,0


In [10]:
data.drop(columns=['result'],inplace=True)

In [11]:
data.head()

,url,label
0,https://www.google.com,benign
1,https://www.youtube.com,benign
2,https://www.facebook.com,benign
3,https://www.baidu.com,benign
4,https://www.wikipedia.org,benign


In [ ]:
type_count = data["label"].value_counts()
type_bar_graph = go.Figure(data=[go.Bar(x=type_count.index,y=type_count,marker={'color':["#37FF00","#FF6600"]})])

type_bar_graph.update_layout(
    xaxis_title='种类',
    yaxis_title='统计数量',
    title='不同url类型的数量',
    plot_bgcolor='black',
    paper_bgcolor='black',
    font=dict(color='white'),
)
type_bar_graph.update_xaxes(tickfont=dict(color='white'))
type_bar_graph.update_yaxes(tickfont=dict(color='white'))
type_bar_graph.show()

## 特征工程

In [ ]:
# 添加特征值：url类型
def label2type(label : str):
    if label == 'benign':
        return 0
    else:
        return 1


data['type'] = data['label'].apply(label2type)

In [16]:
data.head()

,url,label,type
0,https://www.google.com,benign,0
1,https://www.youtube.com,benign,0
2,https://www.facebook.com,benign,0
3,https://www.baidu.com,benign,0
4,https://www.wikipedia.org,benign,0


In [ ]:
def get_url_length(url):
    prefixs = ['http://','https://']
    for prefix in prefixs:
        if prefix in url:
            url = url.replace(prefix,'')
    
    url = url.replace('www.','')
    return len(url)

data['url_length'] = data['url'].apply(get_url_length)

In [72]:
data.head()

,url,label,type,url_length,digits_count,alpha_count,special_char_count
0,https://www.google.com,benign,0,10,0,17,5
1,https://www.youtube.com,benign,0,11,0,18,5
2,https://www.facebook.com,benign,0,12,0,19,5
3,https://www.baidu.com,benign,0,9,0,16,5
4,https://www.wikipedia.org,benign,0,13,0,20,5


In [29]:
# 提取数字，字母，特殊字符特征
def digits_count(url):
    return sum(c.isdigit() for c in url)
data['digits_count'] =data['url'].apply(digits_count)

In [30]:
def alpha_count(url):
    return sum(c.isalpha() for c in url)
data['alpha_count'] = data['url'].apply(alpha_count)

In [31]:
def special_char_count(url):
    return sum(c in string.punctuation for c in url)

data['special_char_count'] = data['url'].apply(special_char_count)

In [32]:
data.head()

,url,label,type,url_length,digits_count,alpha_count,special_char_count
0,https://www.google.com,benign,0,10,0,17,5
1,https://www.youtube.com,benign,0,11,0,18,5
2,https://www.facebook.com,benign,0,12,0,19,5
3,https://www.baidu.com,benign,0,9,0,16,5
4,https://www.wikipedia.org,benign,0,13,0,20,5


In [42]:
data[data['url'].str.contains('?',regex=False)]

,url,label,type,url_length,digits_count,alpha_count,special_char_count
11285,https://www.en.wikipedia.org/wiki/Volney_Ashfo...,benign,0,44,1,44,11
11815,https://www.en.wikipedia.org/wiki?title=Talk:K...,benign,0,47,0,48,11
11816,https://www.en.wikipilipinas.org/index.php?tit...,benign,0,52,0,52,12
11817,https://www.en.wikipilipinas.org/index.php?tit...,benign,0,54,0,55,11
11818,https://www.en.wikipilipinas.org/index.php?tit...,benign,0,55,0,55,12
...,...,...,...,...,...,...,...
632470,www.realmofgaming.com/?page=ps2cheats/bloodray...,malicious,1,65,2,57,10
632476,www.gameindustry.com/review/item.asp?id=613,malicious,1,39,3,33,7
632480,www.game-over.net/reviews.php?page=xboxreviews...,malicious,1,53,3,44,10
632482,www.eurogamer.net/article.php?article_id=61403,malicious,1,42,5,34,7


In [60]:
test_url = "https://www.example.com/path/to/somewhere?q=hello&id=123"
tp = urlparse(test_url)

In [49]:
tp.netloc

'www.example.com'

In [52]:
tp.path

'/path/to/somewhere'

In [61]:
tp.scheme

'https'

In [56]:
tp.query

'q=hello&id=123'

In [78]:
# 提取特征：是否有查询参数，查询参数的长度，网络地址，根域名，url路径

def prase_url(url) -> dict:
    prase = urlparse(url)
    
    
    
    has_query = 1 if prase.query else 0
    
    qs = parse_qs(url)
    query_values = []
    for values in qs.values():
        query_values.extend([v.strip() for v in values if v.strip()])
    # 拼接所有值为一个字符串
    query_values_str = ''.join(query_values)
    
    query_length = len(query_values_str)
    query_alpha_count = alpha_count(query_values_str)
    query_digits_count = digits_count(query_values_str)
    query_special_char_count = special_char_count(query_values_str)
        
        
    
    netloc = prase.netloc
    
    tld = tldextract.extract(url)
    root_domain = tld.domain
    suffix_domain = tld.suffix
    
    is_secure = 1 if prase.scheme == 'https' else 0
    path = prase.path
    fragment = prase.fragment
    url_length_without_query = len(f'{netloc}{fragment}{path}')
    
    return {
        'has_query' : has_query,
        'query_length' : query_length,
        'query_alpha_count':query_alpha_count,
        'query_digits_count':query_digits_count,
        'query_special_char_count':query_special_char_count,
        'netloc' : netloc,
        'root_domain' : root_domain,
        'is_secure' : is_secure,
        'suffix_domain' : suffix_domain,
        'path' : path,
        'url_length_without_query':url_length_without_query
    }

In [82]:
data.loc[11815,'url']

'https://www.en.wikipedia.org/wiki?title=Talk:Kirsten_Costas'

In [83]:
prase_url(data.loc[11815,'url'])

{'has_query': 1,
 'query_length': 19,
 'query_alpha_count': 17,
 'query_digits_count': 0,
 'query_special_char_count': 2,
 'netloc': 'www.en.wikipedia.org',
 'root_domain': 'wikipedia',
 'is_secure': 1,
 'suffix_domain': 'org',
 'path': '/wiki',
 'url_length_without_query': 25}

In [85]:
data.rename(columns={'url_length' : 'url_full_length'},inplace=True)

In [86]:
data['url'].isna().sum()

np.int64(0)

In [87]:
prase_dataframe = data['url'].apply(prase_url)

In [88]:
prase_dataframe

0         {'has_query': 0, 'query_length': 0, 'query_alp...
1         {'has_query': 0, 'query_length': 0, 'query_alp...
2         {'has_query': 0, 'query_length': 0, 'query_alp...
3         {'has_query': 0, 'query_length': 0, 'query_alp...
4         {'has_query': 0, 'query_length': 0, 'query_alp...
                                ...                        
632503    {'has_query': 0, 'query_length': 0, 'query_alp...
632504    {'has_query': 0, 'query_length': 0, 'query_alp...
632505    {'has_query': 0, 'query_length': 0, 'query_alp...
632506    {'has_query': 0, 'query_length': 0, 'query_alp...
632507    {'has_query': 0, 'query_length': 0, 'query_alp...
Name: url, Length: 632508, dtype: object

In [ ]:
# 把字典Series转换成新的DataFrame
df_new_features = prase_dataframe.apply(pd.Series)
df_new_features.head()

,has_query,query_length,query_alpha_count,query_digits_count,query_special_char_count,netloc,root_domain,is_secure,suffix_domain,path,url_length_without_query
0,0,0,0,0,0,www.google.com,google,1,com,,14
1,0,0,0,0,0,www.youtube.com,youtube,1,com,,15
2,0,0,0,0,0,www.facebook.com,facebook,1,com,,16
3,0,0,0,0,0,www.baidu.com,baidu,1,com,,13
4,0,0,0,0,0,www.wikipedia.org,wikipedia,1,org,,17


In [ ]:
# 按照列拼接
data = pd.concat([data,df_new_features],axis=1)

In [98]:
import ipaddress
def have_ip_address(url):
    try:
        parsed_url = urlparse(url)
        # netloc 中剥离端口、用户名密码后的纯域名（或 IP），无则为空
        if parsed_url.hostname:
            # ipaddress.ip_address()：接收字符串参数，若参数是合法的 IPv4/IPv6 地址，则返回对应的 IP 对象（IPv4Address/IPv6Address）；
            # 若不是合法 IP（比如是域名 www.baidu.com），则抛出 ValueError 异常。
            
            # 如'192.168.1.1'
            ip = ipaddress.ip_address(parsed_url.hostname)
            return isinstance(ip, (ipaddress.IPv4Address, ipaddress.IPv6Address))
    except ValueError:
        pass  # 无效域名

    return 0

In [99]:
data['have_id_address'] = data['url'].apply(have_ip_address)

In [100]:
# 检查是否有重复
data.duplicated().sum()

np.int64(0)

In [102]:
data.shape

(632508, 19)

In [103]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 632508 entries, 0 to 632507
Data columns (total 19 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   url                       632508 non-null  str   
 1   label                     632508 non-null  str   
 2   type                      632508 non-null  int64 
 3   url_full_length           632508 non-null  int64 
 4   digits_count              632508 non-null  int64 
 5   alpha_count               632508 non-null  int64 
 6   special_char_count        632508 non-null  int64 
 7   has_query                 632508 non-null  int64 
 8   query_length              632508 non-null  int64 
 9   query_alpha_count         632508 non-null  int64 
 10  query_digits_count        632508 non-null  int64 
 11  query_special_char_count  632508 non-null  int64 
 12  netloc                    632508 non-null  str   
 13  root_domain               632508 non-null  str   
 14  is_secure      

In [105]:
data.columns

Index(['url', 'label', 'type', 'url_full_length', 'digits_count',
       'alpha_count', 'special_char_count', 'has_query', 'query_length',
       'query_alpha_count', 'query_digits_count', 'query_special_char_count',
       'netloc', 'root_domain', 'is_secure', 'suffix_domain', 'path',
       'url_length_without_query', 'have_id_address'],
      dtype='str')

## 图表展示

In [113]:
query_count = data["has_query"].value_counts()
query_count.rename(index={1 :"has query",0  :"no query"},inplace=True)
has_query_pie = go.Figure(go.Pie(labels=query_count.index,values=query_count.values))
has_query_pie.update_layout(title='查询参数情况',
                  template='plotly_dark',
                  font=dict(color='white'),
                  showlegend=True)
has_query_pie.show()

In [114]:

query_count = data["is_secure"].value_counts()
query_count.rename(index={1 :"secure",0  :"no secure"},inplace=True)
has_query_pie = go.Figure(go.Pie(labels=query_count.index,values=query_count.values))
has_query_pie.update_layout(title='查询参数情况',
                  template='plotly_dark',
                  font=dict(color='white'),
                  showlegend=True)
has_query_pie.show()


In [117]:
# Histogram 直方图
# make_subplots(rows=1, cols=1)：创建一个 “子图布局对象”，参数 rows=1, cols=1 表示布局是「1 行 1 列」（即单个图表，无多子图）。
fig = make_subplots(rows=1, cols=1)

# 用 urls_data 中 url_len 列（URL 长度）作为统计对象
# 分为100个区间
# 画出url_len在这100个区间的直方图
fig.add_trace(go.Histogram(x=data['url_full_length'], nbinsx=100))

fig.update_layout(
    title='完整url长度',
    xaxis_title='URL 长度',
    yaxis_title='数量',
    template='plotly_dark',
    font=dict(color='white')
)

fig.show()

In [119]:
# Histogram 直方图
# make_subplots(rows=1, cols=1)：创建一个 “子图布局对象”，参数 rows=1, cols=1 表示布局是「1 行 1 列」（即单个图表，无多子图）。
fig = make_subplots(rows=1, cols=1)

# 用 urls_data 中 url_len 列（URL 长度）作为统计对象
# 分为100个区间
# 画出url_len在这100个区间的直方图
fig.add_trace(go.Histogram(x=data['url_length_without_query'], nbinsx=100))

fig.update_layout(
    title='去除查询参数后url长度',
    xaxis_title='URL 长度',
    yaxis_title='数量',
    template='plotly_dark',
    font=dict(color='white')
)

fig.show()

In [120]:
data.columns

Index(['url', 'label', 'type', 'url_full_length', 'digits_count',
       'alpha_count', 'special_char_count', 'has_query', 'query_length',
       'query_alpha_count', 'query_digits_count', 'query_special_char_count',
       'netloc', 'root_domain', 'is_secure', 'suffix_domain', 'path',
       'url_length_without_query', 'have_id_address'],
      dtype='str')

In [ ]:
# 绘制柱状图
counts = data[['alpha_count', 'digits_count', 'special_char_count']].sum()

fig = go.Figure(data=[
    go.Bar(name='Letters', x=['Count'], y=[counts['alpha_count']]),
    go.Bar(name='Digits', x=['Count'], y=[counts['digits_count']]),
    go.Bar(name='Special Characters', x=['Count'], y=[counts['special_char_count']])
])

fig.update_layout(title='完整url的Letters, Digits, and Special Characters长度',
                  xaxis_title='Type',
                  yaxis_title='Count',
                  barmode='group',
                  template='plotly_dark',
                  font=dict(color='white'))

fig.show()

In [125]:
# 绘制柱状图
data_has_query = data[data["has_query"] == 1]
counts = data_has_query[['query_alpha_count', 'query_digits_count', 'query_special_char_count']].sum()

fig = go.Figure(data=[
    go.Bar(name='Letters', x=['Count'], y=[counts['query_alpha_count']]),
    go.Bar(name='Digits', x=['Count'], y=[counts['query_digits_count']]),
    go.Bar(name='Special Characters', x=['Count'], y=[counts['query_special_char_count']])
])

fig.update_layout(title='查询参数的url的字符统计情况',
                  xaxis_title='Type',
                  yaxis_title='Count',
                  barmode='group',
                  template='plotly_dark',
                  font=dict(color='white'))

fig.show()

In [126]:
# 衡量各个数值特征值的线性关系
numeric_data = data.select_dtypes(include='number')

# 计算数值列的皮尔逊相关系数矩阵（衡量变量间线性相关程度），一个方阵
corr_matrix = numeric_data.corr()

heatmap = go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='Greys',  
)


layout = go.Layout(
    title='Correlation Heatmap (Numeric Values Only)',
    xaxis=dict(title='Variables'),
    yaxis=dict(title='Variables'),
    plot_bgcolor='black', 
    paper_bgcolor='black', 
    font=dict(color='white')  
)


fig = go.Figure(data=heatmap, layout=layout)


fig.show()